In [2]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.ndimage import gaussian_filter
import pandas as pd
import scipy.optimize
import math
import MDAnalysis as md

def weighted_avg_and_std(values, weights):
    """
    Return the weighted average and standard deviation.

    They weights are in effect first normalized so that they 
    sum to 1 (and so they must not all be 0).

    values, weights -- NumPy ndarrays with the same shape.
    """
    average = np.average(values, weights=weights)
    # Fast and numerically precise:
    variance = np.average((values-average)**2, weights=weights)
    return (average, math.sqrt(variance))


path = '/Volumes/Elements/PTM_project/PARK7/TYR_COORD_METAD_chainA_BF40/'

In [4]:
u = md.Universe(path+'processed.pdb',path+'fit1.xtc') 

/Users/olivierstreit/miniconda3/lib/python3.7/site-packages/MDAnalysis/topology/guessers.py:80: UserWarning: Failed to guess the mass for the following atom types: 
  warnings.warn("Failed to guess the mass for the following atom types: {}".format(atom_type))
/Users/olivierstreit/miniconda3/lib/python3.7/site-packages/MDAnalysis/topology/PDBParser.py:330: UserWarning: Element information is absent or missing for a few atoms. Elements attributes will not be populated.
  warnings.warn("Element information is absent or missing for a few "


In [5]:
# pick 5 lowest free energy structures in the open and close state (SASA below or above 0.35 nm^2)

In [29]:

sim=1
T=300
t= 1000
t_discard = 200
frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)


SASA = np.loadtxt(path+'RMSD_SASA_data/sasaTYR67_{}.xvg'.format(sim),skiprows=25)[:,2]
zeta = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,1]
time = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,0]
rbias = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,3]


# FEL rbias reweighting
kbt = 0.008314*T
weights = np.exp((rbias[init_frame:frame]-np.amax(rbias[init_frame:frame]))/kbt)
weights = weights/np.sum(weights)
SASA=SASA[init_frame:frame]
zeta=zeta[init_frame:frame]
time=time[init_frame:frame]


frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)
openframes = np.where(SASA>1.5)[0] 
closedframes = np.where(SASA<0.35)[0] 

# open state lowest free energy
N=5
ind_open = np.argpartition(weights[openframes], -N)[-N:]
# closed state
ind_closed = np.argpartition(weights[closedframes], -N)[-N:]

print(zeta[openframes][ind_open])
print(zeta[closedframes][ind_closed])

print(SASA[openframes][ind_open])
print(SASA[closedframes][ind_closed])


[-9.560890e-01 -2.222625e+00 -1.067000e-03  6.905910e-01 -5.279400e-01]
[-0.120228  3.058406  0.347255 -0.611723 -0.595788]
[1.598 1.546 1.524 1.527 1.509]
[0. 0. 0. 0. 0.]


In [30]:
print(time[openframes][ind_open])
print(time[closedframes][ind_closed])

[406750.01932  404540.019215 956910.045451 636655.030239 388125.018435]
[516015.024509 434040.020616 515450.024483 515970.024507 515960.024507]


In [36]:
print(zeta[openframes][ind_open])
print(zeta[closedframes][ind_closed])

[-9.560890e-01 -2.222625e+00 -1.067000e-03  6.905910e-01 -5.279400e-01]
[-0.120228  3.058406  0.347255 -0.611723 -0.595788]


In [ ]:
print(SASA[openframes][ind_open])
print(SASA[closedframes][ind_closed])

In [34]:


# open states
for TIME in time[openframes][ind_open]:
    FRAME =  int(TIME/5) # 5 ps per frame
    u.trajectory[FRAME]
    protein = u.select_atoms('all')
    with md.Writer('PARK7_chainA_Tyr67_sim{}_open_{}ps.pdb'.format(sim,int(TIME)),protein.n_atoms) as W:
        W.write(protein)

        
print('Done')

Done


In [35]:


# closed states
for TIME in time[closedframes][ind_closed]:
    FRAME =  int(TIME/5) # 5 ps per frame
    u.trajectory[FRAME]
    protein = u.select_atoms('all')
    with md.Writer('PARK7_chainA_Tyr67_sim{}_closed_{}ps.pdb'.format(sim,int(TIME)),protein.n_atoms) as W:
        W.write(protein)

        
print('Done')

Done


In [15]:
# find lowest free energy conformations of proline cis and trans states
sim=1
T=300
t= 1000
t_discard = 200
frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)


SASA = np.loadtxt(path+'RMSD_SASA_data/sasaTYR67_{}.xvg'.format(sim),skiprows=25)[:,2]
zeta = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,1]
time = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,0]
rbias = np.loadtxt(path+'DATA_PLUMED/COLVAR.{}'.format(sim))[:,3]


# FEL rbias reweighting
kbt = 0.008314*T
weights = np.exp((rbias[init_frame:frame]-np.amax(rbias[init_frame:frame]))/kbt)
weights = weights/np.sum(weights)
SASA=SASA[init_frame:frame]
zeta=zeta[init_frame:frame]
time=time[init_frame:frame]


frame = int(t*1000/5)
init_frame = int(t_discard*1000/5)
cisframes = np.where(np.abs(zeta)<1.4)[0] 
transframes = np.where(np.abs(zeta)>1.4)[0] 

# open state lowest free energy
N=5
ind_cis = np.argpartition(weights[cisframes], -N)[-N:]
# closed state
ind_trans = np.argpartition(weights[transframes], -N)[-N:]

print(zeta[cisframes][ind_cis])
print(zeta[transframes][ind_trans])

print(SASA[cisframes][ind_cis])
print(SASA[transframes][ind_trans])



[ 0.595182 -0.120228 -0.595788 -0.611723  0.347255]
[ 2.645841  2.72952  -2.624373 -3.124793  3.058406]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]


In [16]:
print(time[cisframes][ind_cis])
print(time[transframes][ind_trans])

[515375.024479 516015.024509 515960.024507 515970.024507 515450.024483]
[434185.020623 433855.020607 435580.020689 434270.020627 434040.020616]


In [17]:


# cis frames
for TIME in time[cisframes][ind_cis]:
    FRAME =  int(TIME/5) # 5 ps per frame
    u.trajectory[FRAME]
    protein = u.select_atoms('all')
    with md.Writer('PARK7_chainA_Pro66_cis_sim{}_{}ps.pdb'.format(sim,int(TIME)),protein.n_atoms) as W:
        W.write(protein)

        
print('Done')

Done


In [18]:


# trans frames
for TIME in time[transframes][ind_trans]:
    FRAME =  int(TIME/5) # 5 ps per frame
    u.trajectory[FRAME]
    protein = u.select_atoms('all')
    with md.Writer('PARK7_chainA_Pro66_trans_sim{}_{}ps.pdb'.format(sim,int(TIME)),protein.n_atoms) as W:
        W.write(protein)

        
print('Done')

Done
